# Using the Python API
{: .page-header}

## Overview

NekMesh comes equipped with a Python API, which allows Python developers to:

- use and chain together modules that have been written in C++;
- write custom modules from within Python to e.g. perform mesh generation or mesh processinng tasks.

In this tutorial we will give a brief overview of the Python API and how it can be used, motivated by some examples.

## Getting started: Importing the modules

The first step is to `import` the appropriate modules from the `NekPy` package, which is a general set of bindings for the Nektar++ framework.

In [2]:
import NekPy.NekMesh as NekMesh

## Example 1: Creating meshes and elements

In NekMesh, all modules receive a `Mesh` object that is passed between them. This holds at a minimum:

- an integer representing the mesh dimension: e.g. for a triangular mesh, this would be two dimensions;
- an integer representing space dimension, which should be greater than or equal to the mesh dimension;
- a list of the vectors of the desired mesh dimension.

To construct an empty mesh, we create a new `Mesh` object.

In [ ]:
# Create a mesh, and set its space and expansion dimension to 3.
mesh = NekMesh.Mesh()
mesh.expDim = 3
mesh.spaceDim = 3

Now we need to create some elements to go in our mesh. For this, we need to define its nodes using the `Node` class. Nodes are defined using a unique integer ID, and always use three-dimensional coordinates $ (x,y,z) $.

In [ ]:
from NekPy.NekMesh import Node

# Now, construct a list of nodes for this tetrahedron.
tet_nodes = [
    Node(0, 0.0, 0.0, 0.0), Node(1, 1.0, 0.0, 0.0), Node(2, 0.5, 1.0, 0.0),
    Node(3, 0.5, 0.5, 1.0)
]

Now we are in a position to create an element. For this, we need to create an element configuration, which defines:

- the shape type of the element;
- the polynomial order of the element;
- whether the element has high-order face-interior nodes;
- whether the element has high-order volume-interior nodes.

Once we have this, we can create the tetrahedron using the `Element.Create` method. This takes as arguments:

- the element configuration;
- the list of nodes for the element; and
- a list of 'tags' that are associated with the element, which are used to define common regions within the domain -- for example for identifying boundary conditions.

In [ ]:
from NekPy.NekMesh import ElmtConfig, Element
from NekPy.LibUtilities import ShapeType

# Construct a configuration for a simple linear tetrahedron. In this statement,
# the arguments to ElmtConfig are the shape type, the order of the element,
# and whether the has high-order edge nodes and face-interior nodes -- 
# both are False in this case.
tet_config = ElmtConfig(ShapeType.Tetrahedron, 1, False, False)

# Finally, construct the tetrahedron.
tet = Element.Create(tet_config, tet_nodes, [ 0 ])

To put the tetrahedron inside the mesh, we place it inside the `mesh.element` property, which contains lists of the elements of each dimension -- for example, `mesh.element[3]` contains all three-dimensional elements.

In [ ]:
# Add this tetrahedron to our mesh.
mesh.element[3].append(tet)

Finally, we need some boundaries for our tetrahedron! To do this, we pull the nodes from our four faces of the tetrahedron, and construct triangles in the same way, this time putting each triangle inside `mesh.element[2]` as they are two-dimensional elements. Each element gets its own tag running 1 through 4.

In [ ]:
# Now, construct some boundaries for our domain. For this, we need four triangles.
# We extract the nodes for each triangle from the 
tri_config = ElmtConfig(ShapeType.Triangle, 1, False, False)
tri_faces = [ (0,1,2), (0,1,3), (1,2,3), (0,2,3) ]

# Now construct the faces.
for i in range(4):
    mesh.element[2].append(
        Element.Create(
            tri_config, 
            [ tet_nodes[j] for j in tri_faces[i] ],
            [ i+1 ]))

## Example 2: Running modules

Now that we have a mesh, we'd like to be able to output it to some kind of file format. To enable this, we need to first construct one or more `Module`s. The concept of a module is that it should modify the mesh in some manner: to this end, they come in three flavours:

- `InputModule`: constructs a mesh from the ground up, perhaps by creating it analytically or reading a file format.
- `ProcessModule`: alters the mesh in some way to process it: perhaps applying an extrusion, or 
- `OutputModule`: writes the mesh to an output file format.

A typical 'pipeline' of modules therefore has one input module (to read a mesh), zero or more processing modules (to perhaps optimise the mesh in some manner), and one output module (to write a mesh).

NekMesh has a number of built-in C++ modules that are accessible through the Python API. This is beneficial since in this manner, most of the computationally-intensive steps can be performed through C++ and offer significant performance advantages over a purely Python-based approach.

In this example, we'll construct a module, based on the previous example, to set up a mesh with one tetrahedron, and then create a simple pipeline so that we can write our mesh to disk.


### Creating a module
Let's first consider how to create a `Module`. We make use of the same pattern that is used to construct elements: we pass a `Create` function the name of the module we'd like to create, and optionally we can specify configuration parameters for the `Module` to be set up with. To demonstrate this, let's create:

- an output module to write an XML file, which is done using the `OutputModule.Create` function;
- a process module to check the Jacobian of each of our elements.

In [ ]:
from NekPy.NekMesh import OutputModule, ProcessModule

# Create a new, empty mesh.
mesh = NekMesh.Mesh()

# Now create an output module. Notice that we can pass the output filename
# to the module by specifying the 'outfile' keyword argument.
xml_writer = OutputModule.Create('xml', mesh, outfile='single_tet_mesh.xml')

# Similarly, let's create a module to check each element's Jacobian.
jac_module = ProcessModule.Create('jac', mesh, list=True)

Now we have the end of a quick pipeline of modules, let's define our module from within Python. This is a Python class, which should:

- inherit from one of `InputModule`, `OutputModule` or `ProcessModule`, depending on its purpose;
- set up the mesh, in the case of input modules;
- implement a 'Process' function, which takes no arguments and defines the behaviour of the module;
- leave the mesh in a 'completed state': this means that the mesh contains not only the elements, but a set of unique vertices, edges, faces and composites. This is achieved by calling the various Process* functions, which live at the Module level.

To this end, let's implement the code above, to create a single tetrahedron.

In [ ]:
from NekPy.NekMesh import InputModule

class MyInputModule(InputModule):
    def __init__(self, mesh):
        # Make sure to call InputModule's constructor here.
        super().__init__(mesh)

    def Process(self):
        # Set up the mesh expansion and space dimensions.
        self.mesh.expDim = 3
        self.mesh.spaceDim = 3

        # Create tetrahedral nodes & elements, using the code above.
        tet_nodes = [
            Node(0, 0.0, 0.0, 0.0), Node(1, 1.0, 0.0, 0.0), Node(2, 0.5, 1.0, 0.0),
            Node(3, 0.5, 0.5, 1.0)
        ]
        tet_config = ElmtConfig(ShapeType.Tetrahedron, 1, False, False)
        mesh.element[3].append(Element.Create(tet_config, tet_nodes, [ 0 ]))
        
        tri_config = ElmtConfig(ShapeType.Triangle, 1, False, False)
        tri_faces = [ (0,1,2), (0,1,3), (1,2,3), (0,2,3) ]

        # Now construct the faces.
        for i in range(4):
            mesh.element[2].append(
                Element.Create(
                    tri_config, 
                    [ tet_nodes[j] for j in tri_faces[i] ],
                    [ i+1 ]))

        # Call the Module functions to create all of the edges, faces and
        # composites for the mesh.
        self.ProcessVertices()
        self.ProcessEdges()
        self.ProcessFaces()
        self.ProcessElements()
        self.ProcessComposites()

Finally, we need to register our module so that it can be created using `InputModule.Create`.

In [ ]:
from NekPy.NekMesh import Module, ModuleType

# Register this module with our factory.
Module.Register(ModuleType.Input, 'myinputmodule', MyInputModule)

Then we can create our module, and run a simple pipeline! 

In [ ]:
import wurlitzer

mesh.verbose = True
input_module = InputModule.Create('myinputmodule', mesh)

with wurlitzer.sys_pipes():
    # Run each module in succession by calling its Process function.
    input_module.Process()
    jac_module.Process()
    xml_writer.Process()

Finally, we can view the final XML file. Notice that composite 0 contains the tetrahedron, whereas composites 1-4 contain the boundaries.

In [ ]:
with open('single_tet_mesh.xml', 'r') as f:
    print(f.read())